**Data preparation**

In [5]:
import zipfile
import os

zip_filename = "airplanes.zip"
extract_dir = "airplanes_data"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Files extracted to directory: {extract_dir}")
print(os.listdir(extract_dir))


Files extracted to directory: airplanes_data
['Airplanes_Annotations', 'Images']


In [9]:
import os
import shutil
from sklearn.model_selection import train_test_split


annotation_path = "airplanes_data/Airplanes_Annotations/Airplanes_Annotations"
image_path = "airplanes_data/Images/Images"
output_path = "Processed_Dataset"

os.makedirs(output_path, exist_ok=True)

def load_annotations(annotation_dir, image_dir):
    
    data = []

    print(f"Checking annotation directory: {annotation_dir}")
    print(f"Checking image directory: {image_dir}")

    annotation_files = os.listdir(annotation_dir)
    image_files = os.listdir(image_dir)

    print(f"Found {len(annotation_files)} annotation files and {len(image_files)} image files.")

    for csv_file in annotation_files:
        if csv_file.endswith(".csv"):
            annotation_file = os.path.join(annotation_dir, csv_file)
            image_file = os.path.join(image_dir, csv_file.replace(".csv", ".jpg"))

            if not os.path.exists(image_file):
                print(f"Image file missing for {csv_file}, expected: {image_file}")
                continue

           
            try:
                with open(annotation_file, "r") as f:
                    lines = f.readlines()

                    if not lines:
                        print(f"Empty file: {annotation_file}")
                        continue

                   
                    try:
                        airplane_count = int(lines[0].strip()) 
                        print(f"Processing {csv_file}, airplanes: {airplane_count}")
                    except ValueError:
                        print(f"Invalid airplane count in file: {annotation_file}")
                        continue

                  
                    for idx, line in enumerate(lines[1:]):  
                        parts = line.strip().split()  

                      
                        print(f"Line {idx + 1} in {csv_file}: {line.strip()}")
                        print(f"Parsed parts: {parts}")

                        if len(parts) == 4:  
                            try:
                                x_min, y_min, x_max, y_max = map(int, parts)

                                data.append({
                                    "image": image_file,
                                    "bbox": [x_min, y_min, x_max, y_max],
                                    "label": "airplane"
                                })
                                print(f"Added bounding box: {x_min, y_min, x_max, y_max}")
                            except ValueError:
                                print(f"Invalid bounding box values in file: {annotation_file}, line: {line.strip()}")
                        else:
                            print(f"Skipping invalid line in file: {annotation_file}, line: {line.strip()}")
            except Exception as e:
                print(f"Error reading file {annotation_file}: {e}")

    print(f"Total valid annotations: {len(data)}")
    return data

dataset = load_annotations(annotation_path, image_path)


if not dataset:
    print("No data found. Please check the annotation and image directories.")
    exit(1)


train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)

def save_split(data, split_name, output_dir):
   
    split_dir = os.path.join(output_dir, split_name)
    os.makedirs(split_dir, exist_ok=True)

    for item in data:
        image_name = os.path.basename(item["image"])
        label_file = os.path.join(split_dir, image_name.replace(".jpg", ".txt"))

       
        shutil.copy(item["image"], os.path.join(split_dir, image_name))

        with open(label_file, "a") as f:  
            bbox = item["bbox"]
            label = item["label"]
            f.write(f"{label} {bbox[0]} {bbox[1]} {bbox[2]} {bbox[3]}\n")


save_split(train_data, "train", output_path)
save_split(test_data, "test", output_path)

print(f"Data split completed. Processed dataset saved in '{output_path}'")

Streaming output truncated to the last 5000 lines.
Added bounding box: (86, 100, 121, 137)
Line 2 in airplane_346.csv: 29 14 48 34
Parsed parts: ['29', '14', '48', '34']
Added bounding box: (29, 14, 48, 34)
Line 3 in airplane_346.csv: 5 45 25 66
Parsed parts: ['5', '45', '25', '66']
Added bounding box: (5, 45, 25, 66)
Processing airplane_699.csv, airplanes: 3
Line 1 in airplane_699.csv: 12 67 56 102
Parsed parts: ['12', '67', '56', '102']
Added bounding box: (12, 67, 56, 102)
Line 2 in airplane_699.csv: 55 43 93 83
Parsed parts: ['55', '43', '93', '83']
Added bounding box: (55, 43, 93, 83)
Line 3 in airplane_699.csv: 102 14 137 52
Parsed parts: ['102', '14', '137', '52']
Added bounding box: (102, 14, 137, 52)
Processing airplane_585.csv, airplanes: 2
Line 1 in airplane_585.csv: 70 53 128 100
Parsed parts: ['70', '53', '128', '100']
Added bounding box: (70, 53, 128, 100)
Line 2 in airplane_585.csv: 166 146 208 197
Parsed parts: ['166', '146', '208', '197']
Added bounding box: (166, 146,

In [2]:
!pip install torchsummary torchviz

In [33]:

import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.optim as optim
from PIL import Image
import cv2
import numpy as np
from skimage import color
from skimage.segmentation import slic
import torch.nn.functional as F

class AirplaneDataset(Dataset):
    def __init__(self, image_dir="airplanes_data/Images/Images", annotation_dir="airplanes_data/Airplanes_Annotations/Airplanes_Annotations", transform=None):
        self.image_dir = image_dir
        self.annotation_dir = annotation_dir
        self.transform = transform
        self.images = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_name = self.images[idx]
        image_path = os.path.join(self.image_dir, image_name)
        image = Image.open(image_path).convert("RGB")
        annotation_path = os.path.join(self.annotation_dir, image_name.replace(".jpg", ".txt"))
        boxes = []
        labels = []

        with open(annotation_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                label = parts[0]
                x_min, y_min, x_max, y_max = map(int, parts[1:])
                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(0)

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = torch.tensor([idx])

        if self.transform:
            image = self.transform(image)

        return image, target

train_image_dir = 'Processed_Dataset5/train'
train_annotation_dir = 'Processed_Dataset5/train'

transform = transforms.Compose([transforms.ToTensor()])
train_dataset = AirplaneDataset(image_dir=train_image_dir, annotation_dir=train_annotation_dir, transform=transform)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

def simple_segmentation(image):
    image = image.cpu().numpy().transpose((1, 2, 0))
    image = (image * 255).astype(np.uint8)
    segments = slic(image, n_segments=100, compactness=10, sigma=1)
    regions = []
    for label in np.unique(segments):
        segment_mask = (segments == label)
        coords = np.argwhere(segment_mask)
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        regions.append([x_min, y_min, x_max, y_max])
    return regions

class RegionBasedRCNN(torch.nn.Module):
    def __init__(self, backbone, num_classes, resize_dim=(224, 224)):
        super().__init__()
        self.backbone = backbone
        self.classifier = torch.nn.Linear(2048, num_classes)
        self.resize_dim = resize_dim

    def forward(self, images, regions):
        region_features = []
        for image, region_list in zip(images, regions):
            for region in region_list:
                if len(region) == 4:
                    x_min, y_min, x_max, y_max = region
                    region_image = image[:, y_min:y_max, x_min:x_max]
                    region_image_resized = F.interpolate(region_image.unsqueeze(0), size=self.resize_dim)
                    region_features.append(region_image_resized.squeeze(0))
        
        if len(region_features) == 0:
            return torch.tensor([])

        region_features = torch.stack(region_features)
        features = self.backbone(region_features)
        flattened_features = features.flatten(1)
        output = self.classifier(flattened_features)
        return output

backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
backbone = torch.nn.Sequential(*list(backbone.children())[:-1])

model = RegionBasedRCNN(backbone, num_classes=2)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)

def compute_loss(output, targets):
    all_predictions = []
    all_target_labels = []

    for image_idx, (output_image, target) in enumerate(zip(output, targets)):
        num_regions = len(target['boxes'])
        target_labels = target['labels']
        
        target_labels = target_labels.long()

        if num_regions != output_image.size(0):
            output_image = output_image[:num_regions, :]
            target_labels = target_labels[:num_regions]

        all_predictions.append(output_image)
        all_target_labels.append(target_labels)

    all_predictions = torch.cat(all_predictions, dim=0)
    all_target_labels = torch.cat(all_target_labels, dim=0)

    loss = torch.nn.CrossEntropyLoss()(all_predictions, all_target_labels)
    return loss

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for images, targets in train_loader:
        images = [image.to(device) for image in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        region_proposals = [simple_segmentation(image) for image in images]

        optimizer.zero_grad()

        output = model(images, region_proposals)

        loss = compute_loss(output, targets)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss / len(train_loader):.4f}")

def compute_iou(true_boxes, pred_boxes):
    ious = []
    if len(pred_boxes) == 0:
        return [0] * len(true_boxes)

    for true_box in true_boxes:
        ious_per_true_box = []
        for pred_box in pred_boxes:
            x_min1, y_min1, x_max1, y_max1 = true_box
            x_min2, y_min2, x_max2, y_max2 = pred_box

            x_min = max(x_min1, x_min2)
            y_min = max(y_min1, y_min2)
            x_max = min(x_max1, x_max2)
            y_max = min(y_max1, y_max2)

            intersection_area = max(0, x_max - x_min) * max(0, y_max - y_min)
            area_true = (x_max1 - x_min1) * (y_max1 - y_min1)
            area_pred = (x_max2 - x_min2) * (y_max2 - y_min2)
            union_area = area_true + area_pred - intersection_area

            iou = intersection_area / union_area if union_area != 0 else 0
            ious_per_true_box.append(iou)

        if ious_per_true_box:
            ious.append(max(ious_per_true_box))
        else:
            ious.append(0)

    return ious

# Save the trained model
torch.save(model.state_dict(), "rcnn_edited_object_detector.pth")
print("Model saved.")


Epoch 1, Loss: 0.0788
Epoch 2, Loss: 0.0531
Epoch 3, Loss: 0.0505
Epoch 4, Loss: 0.0478
Epoch 5, Loss: 0.0451
Epoch 6, Loss: 0.0435
Epoch 7, Loss: 0.0414
Epoch 8, Loss: 0.0390
Epoch 9, Loss: 0.0361
Epoch 10, Loss: 0.0337
Model saved.


In [32]:
import torch
import numpy as np
from collections import defaultdict

def compute_iou(true_boxes, pred_boxes):
    ious = []

   
    if len(true_boxes) == 0 or len(pred_boxes) == 0:
        return np.array(ious)

    for true_box in true_boxes:
     
        if isinstance(true_box, (np.ndarray, list)) and len(true_box) == 4:
            true_box = np.array(true_box)  
        else:

            continue

        for pred_box in pred_boxes:
        
            if isinstance(pred_box, (np.ndarray, list)) and len(pred_box) == 4:
                pred_box = np.array(pred_box)  
            else:
                continue

           
            x1 = max(true_box[0], pred_box[0])
            y1 = max(true_box[1], pred_box[1])
            x2 = min(true_box[2], pred_box[2])
            y2 = min(true_box[3], pred_box[3])

            intersection_area = max(0, x2 - x1) * max(0, y2 - y1)
            true_area = (true_box[2] - true_box[0]) * (true_box[3] - true_box[1])
            pred_area = (pred_box[2] - pred_box[0]) * (pred_box[3] - pred_box[1])

            union_area = true_area + pred_area - intersection_area
            iou = intersection_area / union_area if union_area > 0 else 0
            ious.append(iou)

    return np.array(ious)



def calculate_ap_at_iou_threshold(true_boxes, pred_boxes, pred_scores, iou_threshold=0.5):
    ious = compute_iou(true_boxes, pred_boxes)
    true_positive = ious >= iou_threshold

    if true_positive.size == 0 or np.sum(true_positive) == 0:
        return 0.0

    sorted_idx = np.argsort(pred_scores)[::-1]
    sorted_true_positive = true_positive[sorted_idx]
    sorted_scores = pred_scores[sorted_idx]

    tp = np.cumsum(sorted_true_positive)
    fp = np.cumsum(1 - sorted_true_positive)

    precision = tp / (tp + fp + np.finfo(float).eps)
    recall = tp / len(true_boxes)

    ap = np.mean(precision)
    return ap

def evaluate_model_on_test(model, data_loader, iou_thresholds=[0.5, 0.75]):
    model.eval()

    all_preds = defaultdict(list)
    all_labels = defaultdict(list)

    with torch.no_grad():
        for images, bboxes, labels in data_loader:
            images = torch.stack([image.to(device) for image in images], dim=0)
            bboxes = [bbox.to(device) for bbox in bboxes]
            labels = [label.to(device) for label in labels]

            pred_bboxes, pred_classes = model(images)

            for bbox, label, pred_bbox, pred_class in zip(bboxes, labels, pred_bboxes, pred_classes):
                true_boxes = bbox.cpu().numpy()
                pred_boxes = pred_bbox.cpu().numpy()

              
                pred_scores = torch.sigmoid(pred_class).cpu().numpy()  

                true_labels = label.cpu().numpy()
                pred_labels = (pred_scores > 0.5).astype(int)  

                for iou_threshold in iou_thresholds:
                    ap = calculate_ap_at_iou_threshold(true_boxes, pred_boxes, pred_scores, iou_threshold)
                    all_preds[iou_threshold].append(ap)
                    all_labels[iou_threshold].append([1] * len(true_boxes))

    ap_results = {}
    for iou_threshold in iou_thresholds:
        ap_results[f'AP@{iou_threshold}'] = np.mean(all_preds[iou_threshold])

    return ap_results

ap_results = evaluate_model_on_test(model, test_loader, iou_thresholds=[0.5, 0.75])

for iou_threshold, ap in ap_results.items():
    print(f"{iou_threshold}: {ap:.4f}")


AP@0.5: 0.6112
AP@0.75: 0.2829
